In [ ]:
# Cài đặt các thư viện cần thiết mà Kaggle có thể thiếu
!pip install -q bs4 pandas huggingface-hub python-dotenv pyyaml pyvi faiss-cpu bm25s

# Cài đặt llama-cpp-python hỗ trợ CUDA (GPU) để LLM chạy nhanh hơn trên Kaggle
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir

In [ ]:
# Thiết lập để import source code từ Kaggle dataset input
import sys
sys.path.append("/kaggle/input/datasets/nostagiguideus17/guru-legal-ai-retrieval")

# Cài đặt đường dẫn làm việc (Kaggle cho phép Ghi/Đọc tại /kaggle/working)
input_path = "/kaggle/working/data"
output_path = "/kaggle/working/output"

In [ ]:
import os
import dataclasses
from pathlib import Path
from huggingface_hub import snapshot_download

from src.config import get_settings, Settings
from src.extraction.pipeline import OCRExtractorPipeline

# Lấy HF_TOKEN từ Kaggle Secrets (Add-ons -> Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Đã nạp HF_TOKEN từ Kaggle Secrets.")
except Exception as e:
    print("Không tìm thấy Kaggle Secrets HF_TOKEN. Hãy thiết lập hoặc sửa trực tiếp trong code.")



settings:Settings = get_settings()

# Ghi đè thuộc tính paths (do Settings sử dụng frozen=True)
# Input Dataset trên Kaggle là Read-Only, ta phải ghi kết quả ra /kaggle/working/
settings.paths = dataclasses.replace(
    settings.paths,
    raw=Path(output_path) / "data" / "raw",
    interim=Path(output_path) / "data" / "interim",
    processed=Path(output_path) / "data" / "processed",
    index=Path(output_path) / "data" / "index",
    questions=Path(input_path) / "data" / "raw" / "questions",
    outputs=Path(output_path) / "data" / "outputs",
    labels=Path(output_path) / "data" / "labels"
)
settings.paths.ensure()

settings.hf_token = hf_token

In [ ]:
if __name__ == "__main__":
    # Tải bộ dataset gốc về thư mục raw (working/data)
    snapshot_download(
        repo_id=settings.dataset,
        repo_type="dataset",
        local_dir=settings.paths.raw,
        token=settings.hf_token
    )
    print("Đã tải dataset thành công. Bắt đầu xử lý OCR...")

    # Khởi chạy Pipeline bóc tách và phân tích dữ liệu
    pipeline = OCRExtractorPipeline(process_dir=settings.paths.raw)
    parsed_documents = pipeline.run()
    print(f"Quy trình OCR hoàn tất. Các tài liệu đã được lưu trữ trong {output_path}.")
    
    # Tiến hành xây dựng toàn bộ Search Index từ documents đã bóc tách
    print("Bắt đầu xây dựng các bộ chỉ mục tìm kiếm (BM25, FAISS HNSW)...")
    from src.retrieval.build import build_all_indexes
    build_all_indexes(parsed_documents)
    print("Toàn bộ quy trình từ Extraction đến Indexing đã hoàn thiện!")
